Importe a  base de dados do Drive

In [2]:
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/Tellus"
AMOSTRA= f"{BASE}/amostras710.csv"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Instale o PyCaret

OBS -> Mudem  o ambiente de excução para o python de 2025/07

In [2]:
pip install pycaret

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 6.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of category-encoders to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of pmdarima to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of pyod to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 7.3 MB/s eta 0:00:00


Primeiro passo é passar a sua base para o pandas

Depois, analise  a base, para saber o q precisa ser ajustado

In [3]:
import pandas as pd

df = pd.read_csv(AMOSTRA)

print("--- Primeiras 5 linhas ---")
display(df.head())

print("\n--- Últimas 5 linhas ---")
display(df.tail())

--- Primeiras 5 linhas ---


,cod_imovel,latitude,longitude,altitude_m,precipitacao_anual_mm,temperatura_media_c,ph_solo,ctc_solo,legenda_solo,mos_solo,declividade_perc,umidade_relativa_perc,radiacao_solar_mj,erosao_classe,distancia_agua_m,cultura_ideal_rotulo
0,AC-1200013-0065DD410D8B4D74884D124552595A75,-10.065699,-67.099232,165.81,1778.0,25.7,4.8,86.0,PVAd,39.0,2.50,84.8,17.1,Média,23.7,Buriti
1,AC-1200013-008511C82A7D491C8FE9A77F156515BC,-9.766742,-67.135746,140.45,1778.0,25.7,4.8,92.0,PVAd,32.0,2.83,84.8,17.0,Média,480.2,Banana
2,AC-1200013-012F582E360D4D95B8EE515A7BFF367D,-9.917798,-66.662608,131.79,1778.0,25.7,4.9,61.0,PVAd,30.0,4.82,84.8,17.0,Média,48.9,Guaraná
3,AC-1200013-01A560FBDDFF49BBA34A961576C7159C,-9.864668,-66.766198,148.72,1778.0,25.7,4.8,81.0,PVAd,35.0,6.60,84.8,17.0,Média,449.7,Cará
4,AC-1200013-01B3514760CF46D28DD72FF7B6E01D3E,-9.697995,-67.129506,130.97,1829.0,25.7,4.8,82.0,PVAd,33.0,2.26,85.1,17.0,Média,264.1,Banana



--- Últimas 5 linhas ---


,cod_imovel,latitude,longitude,altitude_m,precipitacao_anual_mm,temperatura_media_c,ph_solo,ctc_solo,legenda_solo,mos_solo,declividade_perc,umidade_relativa_perc,radiacao_solar_mj,erosao_classe,distancia_agua_m,cultura_ideal_rotulo
704,TO-1700251-0F5AE91371F04AF89749E3D821E339E0,-9.629756,-49.150933,233.64,1617.0,27.3,5.1,69.0,FFc,40.0,2.80,69.2,19.4,Muito alta,182.5,Gengibre
705,TO-1700251-101E7B097D97443AA266549325DB1741,-9.517708,-49.226682,217.25,1617.0,27.3,4.9,73.0,GXbd,40.0,3.73,69.2,19.4,Média,147.8,Gengibre
706,TO-1700251-1029A87BD8D64488AB19BDD20B6A246C,-9.695426,-49.118411,265.17,1617.0,27.3,5.1,66.0,FFc,50.0,2.74,69.2,19.4,Média,392.5,Arroz (sequeiro)
707,TO-1700251-1040BF4854394B6A8A525AAD131B32A5,-9.619685,-49.187555,233.99,1617.0,27.3,5.1,70.0,FFc,38.0,3.20,69.2,19.4,Média,93.9,Gengibre
708,TO-1700251-1041DFD7B0C541B19D05B8098B664C79,-9.695648,-49.220533,233.58,1617.0,27.3,5.0,73.0,FFc,39.0,2.35,69.2,19.4,Média,301.0,Arroz (sequeiro)


Aqui eu defino quem é o X (feature) e quem é o y (target)



In [4]:
FEATURES = [
    "altitude_m",
    "precipitacao_anual_mm",
    "temperatura_media_c",
    "ph_solo",
    "ctc_solo",
    "legenda_solo",
    "mos_solo",
    "declividade_perc",
    "umidade_relativa_perc",
    "radiacao_solar_mj",
    "erosao_classe",
    "distancia_agua_m",
]

TARGET = "cultura_ideal_rotulo"

Aqui eu  converti minhas duas features que estavam em texto para números

In [5]:
from sklearn.preprocessing import LabelEncoder

df_ml = df.copy()

le_solo = LabelEncoder()
le_erosao = LabelEncoder()

df_ml["legenda_solo"] = le_solo.fit_transform(df_ml["legenda_solo"].astype(str))

df_ml["erosao_classe"] = le_erosao.fit_transform(df_ml["erosao_classe"].astype(str))

mapa_solo = dict(zip(le_solo.classes_, le_solo.transform(le_solo.classes_)))
mapa_erosao = dict(zip(le_erosao.classes_, le_erosao.transform(le_erosao.classes_)))

print("\n--- Mapeamento: Legenda Solo ---")
for texto, num in list(mapa_solo.items())[:5]:
    print(f"{num}: {texto}")
print("...")

print("\n--- Mapeamento: Erosão Classe ---")
for texto, num in mapa_erosao.items():
    print(f"{num}: {texto}")

print("\n--- Resumo Final ---")
print("Amostras:", len(df_ml))
print("Classes de cultura mantidas em texto:", df_ml[TARGET].nunique())


--- Mapeamento: Legenda Solo ---
0: AGUA
1: CXbd
2: FFc
3: FXd
4: GXbd
...

--- Mapeamento: Erosão Classe ---
0: Alta
1: Baixa
2: Muito alta
3: Muito baixa
4: Média
5: Área urbana

--- Resumo Final ---
Amostras: 709
Classes de cultura mantidas em texto: 78


Aqui p visualizar a base após a mudança

In [6]:
print("--- Primeiras 5 linhas ---")
display(df_ml.head())

print("\n--- Últimas 5 linhas ---")
display(df_ml.tail())

--- Primeiras 5 linhas ---


,cod_imovel,latitude,longitude,altitude_m,precipitacao_anual_mm,temperatura_media_c,ph_solo,ctc_solo,legenda_solo,mos_solo,declividade_perc,umidade_relativa_perc,radiacao_solar_mj,erosao_classe,distancia_agua_m,cultura_ideal_rotulo
0,AC-1200013-0065DD410D8B4D74884D124552595A75,-10.065699,-67.099232,165.81,1778.0,25.7,4.8,86.0,12,39.0,2.50,84.8,17.1,4,23.7,Buriti
1,AC-1200013-008511C82A7D491C8FE9A77F156515BC,-9.766742,-67.135746,140.45,1778.0,25.7,4.8,92.0,12,32.0,2.83,84.8,17.0,4,480.2,Banana
2,AC-1200013-012F582E360D4D95B8EE515A7BFF367D,-9.917798,-66.662608,131.79,1778.0,25.7,4.9,61.0,12,30.0,4.82,84.8,17.0,4,48.9,Guaraná
3,AC-1200013-01A560FBDDFF49BBA34A961576C7159C,-9.864668,-66.766198,148.72,1778.0,25.7,4.8,81.0,12,35.0,6.60,84.8,17.0,4,449.7,Cará
4,AC-1200013-01B3514760CF46D28DD72FF7B6E01D3E,-9.697995,-67.129506,130.97,1829.0,25.7,4.8,82.0,12,33.0,2.26,85.1,17.0,4,264.1,Banana



--- Últimas 5 linhas ---


,cod_imovel,latitude,longitude,altitude_m,precipitacao_anual_mm,temperatura_media_c,ph_solo,ctc_solo,legenda_solo,mos_solo,declividade_perc,umidade_relativa_perc,radiacao_solar_mj,erosao_classe,distancia_agua_m,cultura_ideal_rotulo
704,TO-1700251-0F5AE91371F04AF89749E3D821E339E0,-9.629756,-49.150933,233.64,1617.0,27.3,5.1,69.0,2,40.0,2.80,69.2,19.4,2,182.5,Gengibre
705,TO-1700251-101E7B097D97443AA266549325DB1741,-9.517708,-49.226682,217.25,1617.0,27.3,4.9,73.0,4,40.0,3.73,69.2,19.4,4,147.8,Gengibre
706,TO-1700251-1029A87BD8D64488AB19BDD20B6A246C,-9.695426,-49.118411,265.17,1617.0,27.3,5.1,66.0,2,50.0,2.74,69.2,19.4,4,392.5,Arroz (sequeiro)
707,TO-1700251-1040BF4854394B6A8A525AAD131B32A5,-9.619685,-49.187555,233.99,1617.0,27.3,5.1,70.0,2,38.0,3.20,69.2,19.4,4,93.9,Gengibre
708,TO-1700251-1041DFD7B0C541B19D05B8098B664C79,-9.695648,-49.220533,233.58,1617.0,27.3,5.0,73.0,2,39.0,2.35,69.2,19.4,4,301.0,Arroz (sequeiro)


Esse momento é crucial para otimização, eu vi quantas amostras de cada cultura eu tinha, como podem ver, eu tinha varias com somente uma amostra,  e isso quebra o pycaret, já que ele tenta sempre pegar  uma quantidade p teste e outra p treino, e somente com uma amostra n tem como

In [7]:
print(df_ml[TARGET].value_counts())

cultura_ideal_rotulo
Mandioca                     46
Cúrcuma (açafrão)            44
Gergelim                     41
Arroz (sequeiro)             36
Banana                       35
                             ..
Girassol                      1
Milho (2ª safra/safrinha)     1
Orégano                       1
Cana-forrageira               1
Quiabo                        1
Name: count, Length: 78, dtype: int64


Então eu simplesmente peguei e apaguei essas culturas com somente uma amostra

In [8]:
contagem_classes = df_ml[TARGET].value_counts()
classes_validas = contagem_classes[contagem_classes >= 2].index
df_ml = df_ml[df_ml[TARGET].isin(classes_validas)]

Só aqui o pycaret entra

Documentação caso queiram: https://pycaret.readthedocs.io/en/latest/index.html

In [ ]:
import pycaret
from pycaret.classification import setup, models,compare_models, pull

setup(
    data=df_ml[FEATURES + [TARGET]],
    target=TARGET,
    session_id=42,
    train_size=0.8,
    fold=5,
    n_jobs=-1,
    use_gpu=False,
    verbose=False,
    fix_imbalance=False,
    remove_multicollinearity=True,
    multicollinearity_threshold=0.9,
    feature_selection=True,
    n_features_to_select=0.8,
    normalize=True,
)

best_models = compare_models(
    sort="F1",
    n_select=len(models()),
    turbo=False,
)

leaderboard = pull()

pd.set_option('display.max_rows', None)

display(leaderboard)

A saída de streaming foi truncada nas últimas 5000 linhas.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wi

,,
,,
Initiated,. . . . . . . . . . . . . . . . . .,12:51:43
Status,. . . . . . . . . . . . . . . . . .,Fitting 5 Folds
Estimator,. . . . . . . . . . . . . . . . . .,Gradient Boosting Classifier


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.5656,0.0000,0.5656,0.5211,0.5213,0.5503,0.5524,2.1700
mlp,MLP Classifier,0.5243,0.0000,0.5243,0.4962,0.4901,0.5086,0.5106,3.7820
dt,Decision Tree Classifier,0.4686,0.0000,0.4686,0.4557,0.4448,0.4519,0.4536,2.3260
gpc,Gaussian Process Classifier,0.4597,0.0000,0.4597,0.3809,0.3953,0.4387,0.4417,10.1520
knn,K Neighbors Classifier,0.4328,0.0000,0.4328,0.3655,0.3769,0.4117,0.4143,2.0580
nb,Naive Bayes,0.3915,0.0000,0.3915,0.3621,0.3562,0.3720,0.3740,2.3920
lr,Logistic Regression,0.3790,0.0000,0.3790,0.2690,0.3009,0.3526,0.3563,4.1400
rbfsvm,SVM - Radial Kernel,0.3825,0.0000,0.3825,0.2475,0.2850,0.3533,0.3590,2.5800
svm,SVM - Linear Kernel,0.2748,0.0000,0.2748,0.2554,0.2296,0.2460,0.2528,2.0080
ridge,Ridge Classifier,0.2639,0.0000,0.2639,0.1321,0.1579,0.2242,0.2337,3.4920


Processing:   0%|          | 0/94 [00:00<?, ?it/s]